In [ ]:
import numpy as np
import pandas as pd
from caveclient import CAVEclient
import copy

from scipy.spatial import cKDTree
import pyvista as pv
from meshparty import meshwork, skeleton_io, trimesh_io
from meshparty.skeleton import Skeleton
import os
import pyfqmr

from cloudvolume import CloudVolume

In [5]:
def denoise_segs(segs):
    #Goes through with a rolling window of 5.
    
    segs_denoise = []

    for i in range(len(segs)):
        temp = segs[i]
        temp['myelin'] = np.where(temp['myelin'] == -1, 0.5, temp['myelin']) #set error entries to 0.5
        temp['myelin'] = np.concatenate(([0], [0], temp['myelin'], [0], [0])) #pad with two zeros on each side
        temp['myelin'] = pd.Series(temp['myelin']).rolling(window=5, center=True).mean().to_numpy() #apply rolling mean with window of 5
        temp['myelin'] = np.where(temp['myelin'] >= 0.5, 1, 0) #threshold at 0.5
        temp['myelin'] = temp['myelin'][2:-2] #remove padding
        segs_denoise.append(temp)
    return segs_denoise

In [6]:
#plot neuron with myelin.

# --------------------------------------
# Step 1: Extract myelin annotations from segments_denoise
# --------------------------------------
def extract_all_myelin_annotations(segments_denoise):
    """
    Args:
        segments_denoise: list of segment dicts with 'pt_position' and 'myelin' keys

    Returns:
        points: (N, 3) np.ndarray of coordinates
        labels: (N,) np.ndarray of 0 or 1
    """
    points = []
    labels = []
    for seg in segments_denoise:
        points.extend(seg['pt_position'])
        labels.extend(seg['myelin'])
    return np.array(points), np.array(labels)


# --------------------------------------
# Step 2: Label mesh vertices with nearest myelin annotation
# --------------------------------------
def label_mesh_vertices_by_nearest_myelin_point(mesh_vertices, annotation_points, annotation_labels):
    """
    Args:
        mesh_vertices: (M, 3)
        annotation_points: (N, 3)
        annotation_labels: (N,) binary

    Returns:
        myelin_labels: (M,) array of 0 or 1
    """
    tree = cKDTree(annotation_points)
    _, indices = tree.query(mesh_vertices, workers=-1)
    return annotation_labels[indices]

# --------------------------------------
# Step 2.5: Map mesh vertices to skeleton vertices
# --------------------------------------


def map_vertices_to_skeleton(mesh_vertices: np.ndarray, skel_vertices: np.ndarray) -> np.ndarray:
    """
    Maps each mesh vertex to the index of the closest skeleton vertex.

    Parameters:
    -----------
    mesh_vertices : np.ndarray
        An (N, 3) array of mesh vertex coordinates.

    skel_vertices : np.ndarray
        An (M, 3) array of skeleton node coordinates.

    Returns:
    --------
    np.ndarray
        An (N,) array of indices, where each element is the index of the closest
        skeleton node to the corresponding mesh vertex.
    """
    if len(skel_vertices) == 0:
        raise ValueError("Skeleton vertices array is empty.")
    if len(mesh_vertices) == 0:
        raise ValueError("Mesh vertices array is empty.")

    tree = cKDTree(skel_vertices)
    _, indices = tree.query(mesh_vertices)
    return indices


# --------------------------------------
# Step 3: Assign base color based on skeleton compartment
# --------------------------------------
def assign_compartment_colors(mesh_vertices, mesh_to_skel_idx, compartments):
    """
    Args:
        mesh_vertices: (M, 3)
        mesh_to_skel_idx: (M,) array mapping mesh vertex to closest skeleton node
        compartments: dict[node_id] -> 1 (soma), 2 (axon), 3 (dendrite)

    Returns:
        base_colors: (M, 3) RGB color array
    """
    base_colors = np.zeros((mesh_vertices.shape[0], 3))
    compartment_labels = np.zeros(mesh_vertices.shape[0], dtype=int)
    for v_idx, skel_node in enumerate(mesh_to_skel_idx):
        label = compartments[skel_node]
        if label == 2:  # axon
            # base_colors[v_idx] = [0.06, 0.48, 0.67]  # blue
            base_colors[v_idx] = [0.5, 0.85, 0.85]  # blue
            compartment_labels[v_idx] = 2
        elif label == 3:  # dendrite
            # base_colors[v_idx] = [139/255, 69/255, 19/255]  # brown
            base_colors[v_idx] = [.5, .5, .5]  # brown
            compartment_labels[v_idx] = 3
        elif label == 1:  # soma
            base_colors[v_idx] = [.5, .5, .5]  # black
            compartment_labels[v_idx] = 1
        else:
            base_colors[v_idx] = [0.7, 0.7, 0.7]  # gray fallback
    return base_colors, compartment_labels


# --------------------------------------
# Step 4: Full Visualization Function
# --------------------------------------
def plot_neuron_with_myelin(mesh, sk_dict, segments_denoise, flip_y=True, inflate_nm=0):
    """
    Args:
        mesh: MeshParty mesh object
        sk_dict: dict with keys 'vertices', 'edges', 'compartment'
        segments_denoise: list of segments with myelin annotations
        flip_y: whether to flip Y-axis for visualization
    """
    # Optional Y-axis flip
    if flip_y:
        mesh.vertices[:, 1] *= -1

    # Create skeleton object
    sk = Skeleton.from_dict(sk_dict)

    # Create Meshwork
    mw = meshwork.Meshwork(mesh=mesh, skeleton=sk)

    # Compartment-based base colors

    mesh_to_skel = map_vertices_to_skeleton(mesh.vertices, sk.vertices)
    # mesh_to_skel = mw.mesh_to_skel_map 

    base_colors, compartment_labels = assign_compartment_colors(mesh.vertices, mesh_to_skel, sk_dict['compartment'])

    # Myelin annotation
    myelin_pts, myelin_labels = extract_all_myelin_annotations(segments_denoise)
    #convert from voxels to nm:
    myelin_pts = myelin_pts * np.array([4, 4, 40])  # Assuming voxel size is [4, 4, 40] nm

    # #print number of myelin labels that are 1
    # print("Number of myelin labels that are 1:", np.sum(myelin_labels == 1))

    if flip_y:
        myelin_pts[:, 1] *= -1
    myelin_mask = label_mesh_vertices_by_nearest_myelin_point(mesh.vertices, myelin_pts, myelin_labels)
    myelin_mask = myelin_mask.astype(bool)

    #set myelin_mask to 0 where compartment label is not 2
    myelin_mask = np.logical_and(myelin_mask, compartment_labels == 2)

    #print number of myelin_mask that are 1
    # print("Number of myelin mask that are 1:", np.sum(myelin_mask == 1))
    # print("base_colors shape:", base_colors.shape)       # (N, 3)
    # print("myelin_mask shape:", myelin_mask.shape)       # (N,) or (N, 1)

    # Override color with myelin color
    # myelin_color = np.array([1.0, 0.2, 0.6])  # hot pink
    myelin_color = np.array([1.0, 0.8, 0.16])  # hot pink

    base_colors[myelin_mask] = myelin_color

    # print("Hot pink count:", np.sum(np.all(base_colors == myelin_color, axis=1)))


    # Build PyVista mesh
    faces_padded = np.concatenate([np.full((mesh.faces.shape[0], 1), 3), mesh.faces], axis=1)
    mesh_poly = pv.PolyData(mesh.vertices, faces=faces_padded)
    mesh_poly.point_data['colors'] = base_colors

    if inflate_nm:
        tmp = mesh_poly.compute_normals(cell_normals=False, point_normals=True, inplace=False)
        tmp.set_active_vectors('Normals', preference='point')
        tmp.point_data['colors'] = mesh_poly.point_data['colors']
        inflated = tmp.warp_by_vector(factor=inflate_nm)
        inflated.point_data['colors'] = tmp.point_data['colors']
        mesh_poly = inflated

    # Plot
    plotter = pv.Plotter()
    plotter.add_mesh(mesh_poly, scalars='colors', rgb=True, opacity=0.75)
    plotter.set_background('black')
    plotter.camera_position = 'zy'
    plotter.show()

In [7]:
client = CAVEclient('minnie65_public')
client.version = 1507
myelin_info_df = pd.read_csv("myelin_info_df.csv")

In [4]:
myelin_info_df[myelin_info_df['cell_type'] == '23P']

,pt_root_id,total_myelin_length,cell_type,number_of_segments
0,864691135938604804,279.49,23P,104
2,864691136196131542,186.98,23P,73
6,864691135258083503,3.13,23P,72
8,864691135274038801,536.44,23P,92
10,864691135325071004,167.33,23P,63
11,864691136674080135,0.00,23P,57
12,864691136034011579,0.00,23P,83
13,864691135447638356,203.07,23P,87
18,864691136053374451,0.00,23P,63
20,864691136108071129,327.04,23P,105


In [34]:
pt_root_id = 864691136674783111 #23P

directory = 'segments_myelin_1507'


sk_df = client.skeleton.get_skeleton(pt_root_id, output_format='swc')
sk_dict = client.skeleton.get_skeleton(pt_root_id, output_format='dict')
sk = Skeleton.from_dict(sk_dict)

#get myelin segments
with open(os.path.join(directory, f'{pt_root_id}.pkl'), 'rb') as f:
    segs = pickle.load(f)
segments_denoise = denoise_segs(segs)

In [35]:
mm = trimesh_io.MeshMeta(
  cv_path=client.info.segmentation_source(),
  disk_cache_path="meshes",
)
# Set the static segmentaiton source to the most recent flat segmentation where that root id is valid
# # seg_source = 'precomputed://gs://iarpa_microns/minnie/minnie65/seg_m1300'
# seg_source = 'precomputed://gs://iarpa_microns/minnie/minnie65/seg_m1412'
# # load from cloudvolume
# cv = CloudVolume(seg_source, progress=False, use_https=True)
# mesh = cv.mesh.get(pt_root_id, lod=3)[pt_root_id]

mesh = mm.mesh(seg_id=pt_root_id, lod = 3)

print(len(mesh.vertices))

2496815


In [36]:
pv.set_jupyter_backend('client')  # or 'static'
# plot_neuron_with_myelin(mesh, sk_dict, segments_denoise, flip_y=False, inflate_nm = 500)
plot_neuron_with_myelin(mesh, sk_dict, segments_denoise, flip_y=False, inflate_nm = 1500)

Widget(value='<iframe src="http://localhost:53172/index.html?ui=P_0x17c9ff710_10&reconnect=auto" class="pyvist…